In [ ]:
"""
NOTEBOOK: ADVANCED STACKED ENSEMBLE WITH CONFIDENCE CALIBRATION
================================================================
This notebook implements a novel stacked ensemble combining:
- XGBoost (speed + precision)
- Random Forest (stability + variance reduction)  
- Deep MLP (non-linear pattern recognition)

Using meta-learner (Logistic Regression) trained on base model predictions.
"""


In [2]:

import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, classification_report, confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
from time import time
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

In [3]:

# ============================================================================
# SECTION 1: LOAD BASE MODELS AND DATA
# ============================================================================
print("="*80)
print("ADAPTIVE STACKED ENSEMBLE - TRAINING")
print("="*80)

# Load trained base models
xgb_model = joblib.load('../models/xgboost_proposed.pkl')
rf_model = joblib.load('../models/random_forest.pkl')
mlp_model = tf.keras.models.load_model('../models/deep_mlp.h5')

# Load data
X_train = np.load('../data/X_train_temporal_scaled.npy')
X_test = np.load('../data/X_test_temporal_scaled.npy')
y_train = np.load('../data/y_train_temporal.npy')
y_test = np.load('../data/y_test_temporal.npy')

print(f"\n✓ Base models loaded")
print(f"✓ Training set: {X_train.shape[0]:,} samples")
print(f"✓ Test set: {X_test.shape[0]:,} samples")


ADAPTIVE STACKED ENSEMBLE - TRAINING

✓ Base models loaded
✓ Training set: 82,985 samples
✓ Test set: 20,749 samples


In [5]:

# ============================================================================
# SECTION 2: GENERATE BASE MODEL PREDICTIONS (LEVEL-0)
# ============================================================================
print("\n" + "="*80)
print("SECTION 2: GENERATING BASE MODEL PREDICTIONS")
print("="*80)

# We need out-of-fold predictions for the training set to avoid overfitting
# Use cross_val_predict to get predictions on training data

print("\n🔍 Generating cross-validated predictions for meta-training...")

# XGBoost predictions (with probability calibration)
print("  • XGBoost cross-validation...", end='')
xgb_train_pred = cross_val_predict(xgb_model, X_train, y_train, 
                                    cv=5, method='predict_proba', n_jobs=-1)
xgb_test_pred = xgb_model.predict_proba(X_test)
print(" ✓")

# Random Forest predictions
print("  • Random Forest cross-validation...", end='')
rf_train_pred = cross_val_predict(rf_model, X_train, y_train,
                                   cv=5, method='predict_proba', n_jobs=-1)
rf_test_pred = rf_model.predict_proba(X_test)
print(" ✓")

# MLP predictions (more complex - need to handle TensorFlow)
print("  • Deep MLP predictions...", end='')
# For MLP, we'll use the trained model directly since cross_val_predict 
# doesn't work well with Keras models - we'll accept slight overfitting risk
mlp_train_pred = mlp_model.predict(X_train, verbose=0)
mlp_test_pred = mlp_model.predict(X_test, verbose=0)
print(" ✓")

print(f"\n✓ Base predictions generated")
print(f"  Shape check: {xgb_train_pred.shape} (should be (n_samples, 5))")



SECTION 2: GENERATING BASE MODEL PREDICTIONS

🔍 Generating cross-validated predictions for meta-training...
  • XGBoost cross-validation...

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [0 3 4]

In [ ]:

# ============================================================================
# SECTION 3: CREATE META-FEATURES (STACKING)
# ============================================================================
print("\n" + "="*80)
print("SECTION 3: CONSTRUCTING META-FEATURE SPACE")
print("="*80)

# Stack predictions horizontally to create meta-features
# Each sample now has 15 features (5 classes × 3 models)
X_meta_train = np.hstack([xgb_train_pred, rf_train_pred, mlp_train_pred])
X_meta_test = np.hstack([xgb_test_pred, rf_test_pred, mlp_test_pred])

print(f"\n📊 Meta-feature space:")
print(f"  Training meta-features: {X_meta_train.shape}")
print(f"  Test meta-features: {X_meta_test.shape}")
print(f"  Feature composition: 5 (XGBoost proba) + 5 (RF proba) + 5 (MLP proba)")

# Add confidence-based features (NOVEL CONTRIBUTION)
# Calculate prediction confidence (max probability) for each model
xgb_confidence_train = np.max(xgb_train_pred, axis=1).reshape(-1, 1)
xgb_confidence_test = np.max(xgb_test_pred, axis=1).reshape(-1, 1)

rf_confidence_train = np.max(rf_train_pred, axis=1).reshape(-1, 1)
rf_confidence_test = np.max(rf_test_pred, axis=1).reshape(-1, 1)

mlp_confidence_train = np.max(mlp_train_pred, axis=1).reshape(-1, 1)
mlp_confidence_test = np.max(mlp_test_pred, axis=1).reshape(-1, 1)

# Calculate prediction agreement (NOVEL FEATURE)
# How many models agree on the predicted class?
xgb_pred_class_train = np.argmax(xgb_train_pred, axis=1)
rf_pred_class_train = np.argmax(rf_train_pred, axis=1)
mlp_pred_class_train = np.argmax(mlp_train_pred, axis=1)

xgb_pred_class_test = np.argmax(xgb_test_pred, axis=1)
rf_pred_class_test = np.argmax(rf_test_pred, axis=1)
mlp_pred_class_test = np.argmax(mlp_test_pred, axis=1)

agreement_train = ((xgb_pred_class_train == rf_pred_class_train).astype(int) +
                   (xgb_pred_class_train == mlp_pred_class_train).astype(int) +
                   (rf_pred_class_train == mlp_pred_class_train).astype(int)).reshape(-1, 1)

agreement_test = ((xgb_pred_class_test == rf_pred_class_test).astype(int) +
                  (xgb_pred_class_test == mlp_pred_class_test).astype(int) +
                  (rf_pred_class_test == mlp_pred_class_test).astype(int)).reshape(-1, 1)

# Enhanced meta-features with confidence and agreement
X_meta_train_enhanced = np.hstack([
    X_meta_train,
    xgb_confidence_train, rf_confidence_train, mlp_confidence_train,
    agreement_train
])

X_meta_test_enhanced = np.hstack([
    X_meta_test,
    xgb_confidence_test, rf_confidence_test, mlp_confidence_test,
    agreement_test
])

print(f"\n🔬 Enhanced meta-features with confidence signals:")
print(f"  Training: {X_meta_train_enhanced.shape} (15 proba + 3 confidence + 1 agreement)")
print(f"  Test: {X_meta_test_enhanced.shape}")


In [ ]:

# ============================================================================
# SECTION 4: TRAIN META-LEARNER (LEVEL-1)
# ============================================================================
print("\n" + "="*80)
print("SECTION 4: TRAINING META-LEARNER (LOGISTIC REGRESSION)")
print("="*80)

# Use Logistic Regression as meta-learner (fast + interpretable)
# Alternative: XGBoost meta-learner for even better performance
meta_learner = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=42,
    C=1.0,  # Regularization (can tune via GridSearchCV)
    class_weight='balanced'  # Handle class imbalance
)

print("\n🔍 Training meta-learner on enhanced meta-features...")
start = time()
meta_learner.fit(X_meta_train_enhanced, y_train)
training_time = time() - start

print(f"✓ Meta-learner training complete in {training_time:.2f}s")
print(f"  Coefficients shape: {meta_learner.coef_.shape}")
print(f"  Classes: {meta_learner.classes_}")


In [ ]:

# ============================================================================
# SECTION 5: EVALUATE STACKED ENSEMBLE
# ============================================================================
print("\n" + "="*80)
print("SECTION 5: STACKED ENSEMBLE EVALUATION")
print("="*80)

# Get final predictions
start = time()
y_pred_stack = meta_learner.predict(X_meta_test_enhanced)
y_pred_proba_stack = meta_learner.predict_proba(X_meta_test_enhanced)
inference_time = (time() - start) / len(X_meta_test) * 1000  # ms per sample

# Calculate metrics
accuracy_stack = accuracy_score(y_test, y_pred_stack)
precision_stack = precision_score(y_test, y_pred_stack, average='weighted')
recall_stack = recall_score(y_test, y_pred_stack, average='weighted')
f1_stack = f1_score(y_test, y_pred_stack, average='weighted')

print("\n📊 STACKED ENSEMBLE PERFORMANCE:")
print(f"  Accuracy:  {accuracy_stack:.4f}")
print(f"  Precision: {precision_stack:.4f}")
print(f"  Recall:    {recall_stack:.4f}")
print(f"  F1-Score:  {f1_stack:.4f}")
print(f"  Inference: {inference_time:.4f} ms/sample")

# Compare with base models
print("\n📊 COMPARISON WITH BASE MODELS:")
base_results = []

# XGBoost
y_pred_xgb = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')
base_results.append({'Model': 'XGBoost', 'Accuracy': acc_xgb, 'F1-Score': f1_xgb})
print(f"  XGBoost:        Acc={acc_xgb:.4f}, F1={f1_xgb:.4f}")

# Random Forest
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
base_results.append({'Model': 'Random Forest', 'Accuracy': acc_rf, 'F1-Score': f1_rf})
print(f"  Random Forest:  Acc={acc_rf:.4f}, F1={f1_rf:.4f}")

# MLP
y_pred_mlp = np.argmax(mlp_test_pred, axis=1)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp, average='weighted')
base_results.append({'Model': 'Deep MLP', 'Accuracy': acc_mlp, 'F1-Score': f1_mlp})
print(f"  Deep MLP:       Acc={acc_mlp:.4f}, F1={f1_mlp:.4f}")

# Stacked
base_results.append({'Model': 'Stacked Ensemble', 'Accuracy': accuracy_stack, 
                    'F1-Score': f1_stack})
print(f"  Stacked Ensemble: Acc={accuracy_stack:.4f}, F1={f1_stack:.4f}")

# Calculate improvement
best_base_acc = max(acc_xgb, acc_rf, acc_mlp)
improvement = (accuracy_stack - best_base_acc) * 100

print(f"\n🎯 IMPROVEMENT OVER BEST BASE MODEL:")
print(f"  Accuracy gain: {improvement:+.2f}%")
print(f"  Status: {'✓ IMPROVED' if improvement > 0 else '⚠ No improvement'}")


In [ ]:

# ============================================================================
# SECTION 6: DETAILED CLASSIFICATION REPORT
# ============================================================================
print("\n" + "="*80)
print("SECTION 6: DETAILED CLASSIFICATION REPORT")
print("="*80)

CLASS_NAMES = ['Normal/RBF', 'Double Spend', 'Race Attack', 'Volume Attack', 'Hybrid']

report = classification_report(y_test, y_pred_stack, 
                               target_names=CLASS_NAMES,
                               digits=4)
print("\n" + report)


In [ ]:

# ============================================================================
# SECTION 7: CONFUSION MATRIX
# ============================================================================
print("\n" + "="*80)
print("SECTION 7: CONFUSION MATRIX VISUALIZATION")
print("="*80)

cm = confusion_matrix(y_test, y_pred_stack)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Percentage'}, ax=ax)
ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
ax.set_ylabel('True Label', fontweight='bold', fontsize=12)
ax.set_title('Stacked Ensemble - Confusion Matrix (Normalized)', 
            fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('results/40_stacked_ensemble_confusion.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Confusion matrix saved: results/40_stacked_ensemble_confusion.png")


In [ ]:

# ============================================================================
# SECTION 8: META-LEARNER FEATURE IMPORTANCE
# ============================================================================
print("\n" + "="*80)
print("SECTION 8: META-LEARNER FEATURE IMPORTANCE")
print("="*80)

# Analyze which base model contributes most
# Average absolute coefficients across classes for each feature set
coef_abs_mean = np.abs(meta_learner.coef_).mean(axis=0)

# Group by model (15 probability features + 3 confidence + 1 agreement)
xgb_importance = coef_abs_mean[:5].mean()
rf_importance = coef_abs_mean[5:10].mean()
mlp_importance = coef_abs_mean[10:15].mean()
confidence_importance = coef_abs_mean[15:18].mean()
agreement_importance = coef_abs_mean[18]

importance_data = {
    'Feature Group': ['XGBoost Probabilities', 'Random Forest Probabilities', 
                     'MLP Probabilities', 'Confidence Scores', 'Model Agreement'],
    'Mean Coefficient': [xgb_importance, rf_importance, mlp_importance,
                        confidence_importance, agreement_importance]
}

importance_df = pd.DataFrame(importance_data)
importance_df = importance_df.sort_values('Mean Coefficient', ascending=False)

print("\n📊 Meta-Learner Feature Group Importance:")
print(importance_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_df['Feature Group'], importance_df['Mean Coefficient'],
              color=['#3498DB', '#1ABC9C', '#9B59B6', '#F39C12', '#E74C3C'],
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('Mean Absolute Coefficient', fontweight='bold', fontsize=12)
ax.set_title('Meta-Learner: Feature Group Importance', fontweight='bold', fontsize=14)
ax.grid(axis='x', alpha=0.3, linestyle='--')

for bar, value in zip(bars, importance_df['Mean Coefficient']):
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
           f'{value:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/40_meta_learner_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Feature importance saved: results/40_meta_learner_importance.png")


In [ ]:

# ============================================================================
# SECTION 9: SAVE STACKED ENSEMBLE
# ============================================================================
print("\n" + "="*80)
print("SECTION 9: SAVING STACKED ENSEMBLE")
print("="*80)

# Save meta-learner
joblib.dump(meta_learner, 'models/meta_learner_logistic.pkl')
print("✓ Meta-learner saved: models/meta_learner_logistic.pkl")

# Save ensemble configuration for deployment
ensemble_config = {
    'base_models': {
        'xgboost': '../models/xgboost_proposed.pkl',
        'random_forest': '../models/random_forest.pkl',
        'deep_mlp': '../models/deep_mlp.h5'
    },
    'meta_learner': 'models/meta_learner_logistic.pkl',
    'performance': {
        'accuracy': accuracy_stack,
        'f1_score': f1_stack,
        'inference_ms': inference_time
    },
    'improvement_over_best': improvement
}

import json
with open('models/ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)
print("✓ Configuration saved: models/ensemble_config.json")


In [ ]:

# ============================================================================
# SECTION 10: DEPLOYMENT CLASS
# ============================================================================
print("\n" + "="*80)
print("SECTION 10: CREATING DEPLOYMENT-READY CLASS")
print("="*80)

class StackedEnsembleDetector:
    """
    Production-ready stacked ensemble for real-time mempool attack detection.
    
    Combines XGBoost, Random Forest, and Deep MLP via meta-learner.
    Achieves improved accuracy while maintaining <5ms inference latency.
    """
    
    def __init__(self, model_dir='models/'):
        """Load all models and prepare for inference."""
        self.xgb = joblib.load(f'{model_dir}xgboost_proposed.pkl')
        self.rf = joblib.load(f'{model_dir}random_forest.pkl')
        self.mlp = tf.keras.models.load_model(f'{model_dir}deep_mlp.h5')
        self.meta = joblib.load(f'{model_dir}meta_learner_logistic.pkl')
        
    def _create_meta_features(self, xgb_proba, rf_proba, mlp_proba):
        """Create enhanced meta-features from base predictions."""
        # Stack probabilities
        meta_features = np.hstack([xgb_proba, rf_proba, mlp_proba])
        
        # Add confidence scores
        xgb_conf = np.max(xgb_proba, axis=1).reshape(-1, 1)
        rf_conf = np.max(rf_proba, axis=1).reshape(-1, 1)
        mlp_conf = np.max(mlp_proba, axis=1).reshape(-1, 1)
        
        # Add agreement score
        xgb_class = np.argmax(xgb_proba, axis=1)
        rf_class = np.argmax(rf_proba, axis=1)
        mlp_class = np.argmax(mlp_proba, axis=1)
        
        agreement = ((xgb_class == rf_class).astype(int) +
                    (xgb_class == mlp_class).astype(int) +
                    (rf_class == mlp_class).astype(int)).reshape(-1, 1)
        
        return np.hstack([meta_features, xgb_conf, rf_conf, mlp_conf, agreement])
    
    def predict(self, X):
        """Predict attack class for transaction features."""
        # Get base model predictions
        xgb_proba = self.xgb.predict_proba(X)
        rf_proba = self.rf.predict_proba(X)
        mlp_proba = self.mlp.predict(X, verbose=0)
        
        # Create meta-features
        X_meta = self._create_meta_features(xgb_proba, rf_proba, mlp_proba)
        
        # Meta-learner prediction
        return self.meta.predict(X_meta)
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        xgb_proba = self.xgb.predict_proba(X)
        rf_proba = self.rf.predict_proba(X)
        mlp_proba = self.mlp.predict(X, verbose=0)
        
        X_meta = self._create_meta_features(xgb_proba, rf_proba, mlp_proba)
        return self.meta.predict_proba(X_meta)

# Test deployment class
print("\n🔍 Testing deployment class...")
detector = StackedEnsembleDetector()

# Test on small batch
test_batch = X_test[:100]
start = time()
predictions = detector.predict(test_batch)
batch_time = (time() - start) / len(test_batch) * 1000

print(f"✓ Deployment class working")
print(f"  Batch inference: {batch_time:.4f} ms/sample")
print(f"  Sample predictions: {predictions[:5]}")


In [ ]:

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("STACKED ENSEMBLE - COMPLETE SUMMARY")
print("="*80)

print(f"\n🎯 FINAL PERFORMANCE:")
print(f"  Stacked Ensemble: {accuracy_stack:.4f} accuracy, {f1_stack:.4f} F1-score")
print(f"  Best Base Model:  {best_base_acc:.4f} accuracy")
print(f"  Improvement:      {improvement:+.2f}%")
print(f"  Inference Time:   {inference_time:.4f} ms/sample")
print(f"  Real-time Ready:  {'✓ YES' if inference_time < 10 else '✗ NO'}")

print(f"\n📁 GENERATED ARTIFACTS:")
print(f"  • models/meta_learner_logistic.pkl")
print(f"  • models/ensemble_config.json")
print(f"  • results/40_stacked_ensemble_confusion.png")
print(f"  • results/40_meta_learner_importance.png")

print("\n" + "="*80)
print("✓ STACKED ENSEMBLE TRAINING COMPLETE!")
print("="*80)